# myfpo online

Clone the repo, install it, run the Hopper feasibility gate.

In [ ]:
import os

OUTPUT = {"colab": "/content/", "kaggle": "/kaggle/working/", "local": "./"}

def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    return "kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "local"

def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata
            return userdata.get(key_name)
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key_name)
        return os.getenv(key_name)
    except Exception as exc:
        print(f"load_secret({key_name}) failed: {exc}")
        return None

ENV_NAME = detect_env()
print(f"Env: {ENV_NAME}")

In [ ]:
%cd $OUTPUT

!rm -rf myfpo

token = load_secret("GITHUB_TOKEN")
slug = os.getenv("GITHUB_REPO", "dzungphieuluuky/myfpo")
url = f"https://github.com/{slug}.git"
if token:
    url = url.replace("github.com", f"{token}@github.com")
print("Clone:", url.replace(token or "", "<TOKEN>"))

!git clone $url myfpo
%cd myfpo
!pwd

In [ ]:
!uv pip install torch numpy einx draccus gymnasium gymnasium-robotics mujoco pyyaml tensorboard wandb

In [ ]:
# Scripts import the myfpo package; a fresh clone needs an install.
import subprocess
import sys

rc = subprocess.call(["uv", "pip", "install", "-e", ".", "--no-deps"])
if rc:
    rc = subprocess.call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
if rc:
    raise SystemExit("editable install of myfpo failed; see the log above")
print("myfpo installed editable")

In [ ]:
# Guard: every later cell needs this import to resolve.
import sys
subprocess.run([sys.executable, "-c", "import myfpo, myfpo.training"], check=True)

In [ ]:
key = load_secret("WANDB_API_KEY")
if key:
    !wandb login $key
else:
    print("No WANDB_API_KEY; runs stay offline.")

## Hopper feasibility gate

Does the current method learn on Hopper-v5? Seeds 3407,1,2.
`FEAS_STEPS` is the pilot size; raise it only on the cluster.

- `fpo_pp`: the method as shipped. No flags.
- `fpo_chunk_avg`: plain FPO control (one loss per chunk).

In [ ]:
FEAS_ENV = "Hopper-v5"
FEAS_STEPS = 300_000
FEAS_SEEDS = "3407,1,2"
FEAS_WANT = len(FEAS_SEEDS.split(","))

FEAS_ARMS = {
    "fpo_pp": [],
    "fpo_chunk_avg": ["--myfpo.do_average_cfm_loss_in_chunk", "true"],
}

In [ ]:
# Shared helpers for every arm.
import csv
import glob
import os
import statistics as st
import subprocess

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MUJOCO_GL", "egl")

def read_rows(tag):
    """Summary rows of one arm; [] when the file is absent."""
    path = f"outputs/{tag}/sweep_summary.csv"
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

def sweep(tag, extra):
    """Run all seeds for one arm; skip if the summary is full."""
    if len(read_rows(tag)) >= FEAS_WANT:
        print(f"{tag}: done, skipping")
        return
    cmd = ["python", "scripts/multiseed.py", "run", "--env", FEAS_ENV,
           "--seeds", FEAS_SEEDS, "--total_steps", str(FEAS_STEPS),
           "--out", f"outputs/{tag}", "--method", "myfpo", *extra]
    print(f"\n=== {tag} ===\n{' '.join(cmd)}\n", flush=True)
    print(f"=== {tag} exit={subprocess.call(cmd)} ===", flush=True)

def median_last10(tag):
    return [float(r["reward_median_last10"]) for r in read_rows(tag)]

def pooled_se(a, b):
    ses = [st.stdev(v) / len(v) ** 0.5 for v in (a, b) if len(v) > 1]
    return sum(s * s for s in ses) ** 0.5 if ses else float("nan")

def tensorboard_last(run_dir, keys):
    """Last value of the first matching scalar under run_dir/tensorboard."""
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    for tb in glob.glob(os.path.join(run_dir, "tensorboard", "*")):
        acc = EventAccumulator(tb, size_guidance={"scalars": 0})
        acc.Reload()
        tags = acc.Tags()["scalars"]
        for key in keys:
            if key in tags:
                vals = acc.Scalars(key)
                if vals:
                    return float(vals[-1].value)
    return float("nan")

In [ ]:
for tag, extra in FEAS_ARMS.items():
    sweep(tag, extra)

In [ ]:
# Gate table: row count (exit 0), random baseline, pooled-SE gap.
scores = {tag: median_last10(tag) for tag in FEAS_ARMS}
for tag, fs in scores.items():
    if len(fs) < FEAS_WANT:
        print(f"{tag:>14}  ({len(fs)}/{FEAS_WANT} seeds)")
        continue
    sd = st.stdev(fs) if len(fs) > 1 else float("nan")
    print(f"{tag:>14}  n={len(fs)}  last10={st.fmean(fs):.1f}  sd={sd:.1f}")

fp = scores.get("fpo_pp")
if fp:
    rows = read_rows("fpo_pp")
    base = st.fmean([tensorboard_last(r["run_dir"], ["eval/random_policy_return"]) for r in rows])
    pmean = st.fmean(fp)
    print(f"\nfpo_pp {pmean:.1f} vs random {base:.1f}")
    print("GATE 2 PASS" if pmean > base else "GATE 2 FAIL: stop, do not scale")

if all(len(v) >= FEAS_WANT for v in scores.values()):
    a, b = scores["fpo_pp"], scores["fpo_chunk_avg"]
    gap, se = st.fmean(a) - st.fmean(b), 2 * pooled_se(a, b)
    ok = abs(gap) > se
    print(f"fpo_pp - fpo_chunk_avg = {gap:+.1f}  (2x pooled SE {se:.1f}; "
          + ("outside noise" if ok else "inside noise") + ")")

In [ ]:
# Noise floor of fpo_pp. A gain below it decides nothing.
ckpts = sorted(glob.glob("outputs/fpo_pp/*/checkpoints/best"))
if not ckpts:
    print("no fpo_pp best checkpoint yet; run the sweep cell first")
else:
    subprocess.call(["python", "scripts/determinism_probe.py", "--env", FEAS_ENV,
                     "--checkpoint", ckpts[0], "--episodes", "50", "--repeats", "3"])

## Gate

1. Both arms exit 0; one row per seed.
2. `fpo_pp` beats the random baseline.
3. `fpo_pp` beats `fpo_chunk_avg` by more than 2x the pooled SE and more than
   the probe floor.

All pass: scale up. Any fail: stop. Watch `train/cfm_both_clamped_frac` and
`train/log_ratio_clip_frac` on wandb.